In [ ]:
import os, pickle, sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path

CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    PROJECT_ROOT = CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from mechanisms.CoP.cop import CoP_Mechanism
from utils.cmf_pmf_cal import get_cmf_pmf_dict_with_alphabet
from utils.data_perturb import Perturbation
from utils.pmi_cal import get_pmi_dict
from utils.util_functions import validate_perturbed_data
from utils.eval_perturbed import Evaluate_Perturbed_Data
from utils.mi_compute import get_mi_dict

In [ ]:

THR = 0.5
THR_mi = 0.0
ALPHA = 0.1

In [ ]:
# Privacy budgets used for perturbation.
# Larger epsilon means weaker privacy but usually better utility.
EPS_ARRAY = [1, 2, 4, 10, 15]

# Define input dataset location and output prefix.
# The perturbed outputs will be saved under results/dummy_perturbed_eps_<eps>.csv.zip
FILE_LOCATION_LOAD = f"{PROJECT_ROOT}/datasets/dummy.csv"
FILE_LOCATION_SAVE = f"{PROJECT_ROOT}/results/dummy"

# Load the dataset.
# Each row is a multidimensional user record and each column is an attribute.
data = pd.read_csv(FILE_LOCATION_LOAD, sep=',', engine='python', dtype=int)

# Remove missing values, if any.
# This ensures that dependency estimation and perturbation are performed on clean records.
data = data.replace('?', pd.NA).dropna()
data.dropna(inplace=True)

# Store basic dataset information.
data_samples = data.shape[0]
COLUMNS = data.columns.to_list()

# Estimate prior dependency information from the original dataset.
# CMF_dict stores conditional probability matrices between attribute pairs.
# alphabet_dict stores the domain/alphabet of each attribute.
CMF_dict, alphabet_dict = get_cmf_pmf_dict_with_alphabet(
    data=data,
    is_pmf=False
)

# Compute pairwise mutual information values.
# These values are used by CoP to identify strongly dependent attribute pairs.
mi_dict = get_mi_dict(data=data, COLUMNS=COLUMNS)

# Compute pointwise mutual information values.
# These values are used to identify strong value-level dependencies.
PMI_dict = get_pmi_dict(data=data)

# Initialize the CoP mechanism.
# CoP uses the dependency information above to construct coordinated
# perturbation mechanisms for multidimensional data.
cop_mechanism = CoP_Mechanism(
    ordered_attribute_list=COLUMNS,
    alphabet_dict=alphabet_dict,
    CMF_dict=CMF_dict,
    pmi_dict=PMI_dict,
    mi_dict=mi_dict
)

# Create the perturbation pipeline.
# This object applies the selected privacy mechanism to the input dataset
# and saves perturbed datasets for each epsilon value.
perturbation = Perturbation(
    data=data[COLUMNS],
    mechanism=cop_mechanism,
    COLUMNS=COLUMNS,
    save_location=FILE_LOCATION_SAVE
)

# Run perturbation for all privacy budgets in EPS_ARRAY.
# This generates one perturbed dataset for each epsilon value.
perturbation.perturb(EPS_ARRAY=EPS_ARRAY)

# Load the generated perturbed datasets for validation and evaluation.
perturbed_data_list = []

for eps in EPS_ARRAY:
    perturbed_data_location = f"{FILE_LOCATION_SAVE}_perturbed_eps_{eps}.csv.zip"

    perturbed_data = pd.read_csv(
        perturbed_data_location,
        compression='zip',
        engine='python',
        dtype=str
    )[COLUMNS]

    perturbed_data_list.append(perturbed_data)
    print(f'{perturbed_data_location} loaded.')

# Validate that the generated perturbed datasets have the expected shape
# and contain the required attributes.
original_len = data.shape[0]
validate_perturbed_data(original_len, COLUMNS, perturbed_data_list)

# Compute how many perturbed copies were generated per original record.
# This is useful when the perturbation process produces multiple samples
# for each privacy budget.
number_of_copies = int(
    np.shape(perturbed_data_list[0].values)[0] / original_len
)

# Initialize the evaluation module.
# This compares the perturbed datasets with the original data to measure
# the utility of the perturbed outputs.
evaluation = Evaluate_Perturbed_Data(
    original_data=data[COLUMNS],
    EPS_ARRAY=EPS_ARRAY,
    COLUMNS=COLUMNS,
    perturbed_data_list=perturbed_data_list,
    save_location=FILE_LOCATION_SAVE,
    alphabet_dict=alphabet_dict,
    numerical=np.ones(len(COLUMNS))
)

# Evaluate the perturbed datasets.
# The evaluation results are saved using the provided save_location prefix.
evaluation.evaluate_perturbed_data()

print("Perturbed datasets are generated, validated, and evaluated successfully.")

In [ ]:
mse_loss = []

for eps in EPS_ARRAY:
    with open(f"{PROJECT_ROOT}/results/dummy_utility_evaluation_eps_{eps}.pkl", 'rb') as file:
        mse_loss.append(pickle.load(file)["_0_1_loss"])

plt.figure(figsize=(4, 3))

plt.plot(EPS_ARRAY, mse_loss)
plt.legend(['CoP'], fontsize=10)

plt.xlabel(r"$\operatorname{Privacy \ Budget}$ $(\epsilon)$", fontsize=16)
plt.ylabel(r"$\operatorname{MSE}$", fontsize=16)
plt.tick_params(axis='x', labelsize=12)
plt.tick_params(axis='y', labelsize=12) 